# Chat Completions Providers

The `chat_completions` provider talks to any OpenAI-compatible Chat Completions endpoint. Register an `AsyncOpenAI` client under `"chat_completions"`, then select models with the `chat_completions/` prefix (the remainder is the API model id, and may itself contain `/`).


In [1]:
import os

from openai import AsyncOpenAI
from openai.types.responses import EasyInputMessageParam

from interop_router.router import Router
from interop_router.types import ChatMessage, RouterResponse


def extract_text(response: RouterResponse) -> str:
    texts: list[str] = []
    for chat_message in response.output:
        if chat_message.message.get("type") == "message":
            content = chat_message.message.get("content")
            if isinstance(content, list):
                for c in content:
                    text = c.get("text", "")
                    if text:
                        texts.append(text)
    return "\n".join(texts)


message = ChatMessage(message=EasyInputMessageParam(role="user", content="Hello! Reply in one short sentence."))

## OpenAI

Use a default `AsyncOpenAI()` client (reads `OPENAI_API_KEY`). Prefix the model so the router uses Chat Completions instead of the Responses API.


In [2]:
openai_router = Router()
openai_router.register("chat_completions", AsyncOpenAI())

response = await openai_router.create(
    input=[message],
    model="chat_completions/gpt-5.6-luna",
)
print(extract_text(response))

Hello! How can I help you today?


## Local vLLM

Point `base_url` at a local OpenAI-compatible server. Set `CHAT_COMPLETIONS_BASE_URL` and `CHAT_COMPLETIONS_MODEL` in `.env` (see `.env.example`). The model id after `chat_completions/` is passed through unchanged.


In [3]:
base_url = os.environ["CHAT_COMPLETIONS_BASE_URL"]
model = os.environ["CHAT_COMPLETIONS_MODEL"]

vllm_router = Router()
vllm_router.register("chat_completions", AsyncOpenAI(base_url=base_url))

response = await vllm_router.create(
    input=[message],
    model=f"chat_completions/{model}",
)
print(extract_text(response))



Hi there, how can I assist you today?


## OpenRouter

OpenRouter exposes an OpenAI-compatible Chat Completions API. Set `base_url` and provide an `OPENROUTER_API_KEY`.


In [4]:
openrouter_router = Router()
openrouter_router.register(
    "chat_completions",
    AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    ),
)

response = await openrouter_router.create(
    input=[message],
    model="chat_completions/nvidia/nemotron-3-ultra-550b-a55b:free",
)
print(extract_text(response))

Hello! How can I help you today?


## Token counting

Chat Completions has no native count endpoint. InteropRouter estimates prompt tokens with tiktoken using OpenAI Chat Completions framing (not arbitrary chat templates such as Llama on vLLM).

The model id after `chat_completions/` is treated as a tiktoken encoding name (`o200k_base`, `cl100k_base`, ...) or an OpenAI model id that tiktoken knows. Unknown values fall back to `o200k_base`. Counting is local; no API call is made.


In [5]:
from openai.types.responses.function_tool_param import FunctionToolParam

# Client is unused: chat_completions count_tokens is local.
token_router = Router()
token_router.register("chat_completions", AsyncOpenAI())

get_weather = FunctionToolParam(
    type="function",
    name="get_weather",
    description="Get the current weather in a location",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
    strict=True,
)

base = await token_router.count_tokens(
    input=[message],
    model="chat_completions/gpt-5.6-luna",
)
with_instructions = await token_router.count_tokens(
    input=[message],
    model="chat_completions/nvidia/Qwen3.6-27B-NVFP4",
    instructions="Always answer in one short sentence.",
)
with_tools = await token_router.count_tokens(
    input=[message],
    model="chat_completions/nvidia/Qwen3.6-27B-NVFP4",
    tools=[get_weather],
)
# Prefer an explicit encoding when the API model id is unknown to tiktoken.
with_encoding = await token_router.count_tokens(
    input=[message],
    model="chat_completions/o200k_base",
)

print(f"base: {base}")
print(f"with instructions: {with_instructions}")
print(f"with tools: {with_tools}")
print(f"explicit encoding: {with_encoding}")

base: 15
with instructions: 26
with tools: 54
explicit encoding: 15
